---
# Practical 2: Connecting neurons and measuring synaptic strength

## One presynaptic neuron, several postsynaptic neurons

We create a presynaptic `iaf_psc_alpha` neuron and make it fire **exactly one spike** with a brief strong current pulse. It is connected to six postsynaptic neurons, each with a different synaptic weight (negative = inhibitory). We record the membrane potential *and* the synaptic currents `I_syn_ex`, `I_syn_in` of the postsynaptic neurons.

The "strength" of a synapse can be measured at two levels:
- **PSC amplitude** (pA) — what NEST calls the weight for this model;
- **PSP amplitude** (mV) — what an electrophysiologist would record at the soma.

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

weights = [-200.0, -100.0, 50.0, 100.0, 200.0, 400.0]   # pA
syn_delay = 1.5                                         # ms
t_sim = 300.0

pre = nest.Create("iaf_psc_alpha")
kick = nest.Create("dc_generator", params={"amplitude": 3000.0, "start": 100.0, "stop": 103.0})
nest.Connect(kick, pre)

post = nest.Create("iaf_psc_alpha", len(weights))
for i in range(len(weights)):
    nest.Connect(pre, post[i], syn_spec={"synapse_model": "static_synapse",
                                         "weight": weights[i], "delay": syn_delay})

voltmeter = nest.Create("voltmeter", params={"interval": 0.1, "record_from": ["V_m", "I_syn_ex", "I_syn_in"]})
spike_rec_pre = nest.Create("spike_recorder")
nest.Connect(voltmeter, post)
nest.Connect(pre, spike_rec_pre)

nest.Simulate(t_sim)

t_pre_spikes = spike_rec_pre.get("events")["times"]
print("Presynaptic spike times:", t_pre_spikes)

Before measuring, let's check what NEST actually created. `GetConnections` returns the synapses, and we can read their parameters:

In [ ]:
conns = nest.GetConnections(source=pre, target=post)
for src, tgt, w, d, model in zip(conns.source, conns.target, conns.weight, conns.delay, conns.synapse_model):
    print(f"{src} -> {tgt}   weight = {w:7.1f} pA   delay = {d:.1f} ms   model = {model}")

In [ ]:
vm_ev = voltmeter.get("events")
t_spike = t_pre_spikes[0]

psc_peaks, psp_amps, psp_latencies = [], [], []
fig, (ax_i, ax_v) = plt.subplots(1, 2, figsize=(11, 4))

for gid, w in zip(post.tolist(), weights):
    mask = vm_ev["senders"] == gid
    t = vm_ev["times"][mask]
    v = vm_ev["V_m"][mask]
    i_syn = vm_ev["I_syn_ex"][mask] + vm_ev["I_syn_in"][mask]

    baseline = v[t < t_spike].mean()
    dv = v - baseline
    idx = np.argmax(np.abs(dv))
    psp_amps.append(dv[idx])
    psp_latencies.append(t[idx] - t_spike)
    psc_peaks.append(i_syn[np.argmax(np.abs(i_syn))])

    window = (t > t_spike - 5) & (t < t_spike + 60)
    ax_i.plot(t[window] - t_spike, i_syn[window], label=f"w = {w:.0f} pA")
    ax_v.plot(t[window] - t_spike, dv[window], label=f"w = {w:.0f} pA")

ax_i.set_xlabel("time after presynaptic spike (ms)"); ax_i.set_ylabel("I_syn (pA)"); ax_i.set_title("PSC")
ax_v.set_xlabel("time after presynaptic spike (ms)"); ax_v.set_ylabel("ΔV_m (mV)");  ax_v.set_title("PSP")
ax_v.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(" weight (pA) | PSC peak (pA) | PSP amp (mV) | time to PSP peak (ms)")
for w, ipk, a, lat in zip(weights, psc_peaks, psp_amps, psp_latencies):
    print(f"   {w:7.1f}   |   {ipk:8.1f}    |   {a:7.3f}    |   {lat:5.1f}")

**Synaptic efficacy.** For a passive LIF membrane the PSP is a linear function of the weight. We fit a line to obtain the efficacy in mV per pA of PSC peak.

In [ ]:
slope, intercept = np.polyfit(weights, psp_amps, 1)
print(f"PSP efficacy: {slope*1000:.2f} µV per pA   (intercept {intercept:.1e} mV)")
print(f"-> a 100 pA synapse gives a PSP of {slope*100:.2f} mV")
print(f"-> naive estimate of synchronous 100 pA inputs to reach threshold: {(V_th - E_L) / (slope*100):.1f}")

w_fine = np.linspace(min(weights), max(weights), 100)
plt.figure(figsize=(5, 4))
plt.plot(weights, psp_amps, "o")
plt.plot(w_fine, slope * w_fine + intercept, "k--", lw=0.8)
plt.axhline(0, c="grey", lw=0.5); plt.axvline(0, c="grey", lw=0.5)
plt.xlabel("synaptic weight (pA)")
plt.ylabel("PSP amplitude (mV)")
plt.title("Weight → PSP (iaf_psc_alpha)")
plt.show()

## Convergence: how many synchronous inputs are needed?

We now create a population of presynaptic neurons that all fire together (same pulse), and postsynaptic neurons that receive input from $k = 1, 2, \dots$ of them. We measure the peak depolarisation and whether the postsynaptic neuron fires, and compare with the linear estimate from 2.1.

In [1]:
nest.ResetKernel()
nest.resolution = 0.1

n_pre = 30
w_conv = 100.0     # pA
k_values = list(range(1, n_pre + 1))

pre_pop = nest.Create("iaf_psc_alpha", n_pre)
kick = nest.Create("dc_generator", params={"amplitude": 3000.0, "start": 100.0, "stop": 103.0})
nest.Connect(kick, pre_pop)

post_pop = nest.Create("iaf_psc_alpha", len(k_values))
for i in range(len(k_values)):
    nest.Connect(pre_pop[:k_values[i]], post_pop[i], syn_spec={"weight": w_conv, "delay": 1.5})

voltmeter = nest.Create("voltmeter", params={"interval": 0.1})
spike_rec = nest.Create("spike_recorder")
nest.Connect(voltmeter, post_pop)
nest.Connect(post_pop, spike_rec)
nest.Simulate(200.0)

vm_ev = voltmeter.get("events")
spiking_senders = spike_rec.get("events")["senders"]

peak_v, fired = [], []
for gid in post_pop.tolist():
    mask = vm_ev["senders"] == gid
    peak_v.append(vm_ev["V_m"][mask].max())
    fired.append(gid in spiking_senders)
peak_v = np.array(peak_v)
fired = np.array(fired)

k_min = k_values[np.argmax(fired)] if fired.any() else None
print(f"Minimal number of synchronous inputs to fire: k = {k_min}")
print(f"Linear prediction from 2.1:                  k = {(V_th - E_L) / (slope * w_conv):.1f}")

plt.figure(figsize=(6, 4))
plt.plot(np.array(k_values)[~fired], peak_v[~fired], "o", label="no spike")
plt.plot(np.array(k_values)[fired], np.full(fired.sum(), V_th), "r^", label="spike (V reset)")
plt.axhline(V_th, ls="--", c="grey", lw=0.8)
plt.xlabel("number of synchronous inputs k")
plt.ylabel("peak V_m (mV)")
plt.legend()
plt.show()

NameError: name 'nest' is not defined

In the LIF model PSPs add **perfectly linearly**, so the prediction is exact. Keep this in mind for the HH comparison notebook.

## Temporal summation

Now a single presynaptic neuron fires repetitively. Each presynaptic neuron gets a different bias current (hence a different rate), and drives its own postsynaptic neuron through the same synapse. When successive PSPs arrive faster than the membrane can decay ($\tau_m$), they summate.

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

pre_currents = [400.0, 600.0, 1000.0, 1500.0]   # pA -> different presynaptic rates
w_sum = 600.0                                  # pA
t_sim = 500.0

pre_pop = nest.Create("iaf_psc_alpha", len(pre_currents))
pre_pop.set(I_e=pre_currents)
post_pop = nest.Create("iaf_psc_alpha", len(pre_currents))
nest.Connect(pre_pop, post_pop, "one_to_one", syn_spec={"weight": w_sum, "delay": 1.5})

voltmeter = nest.Create("voltmeter", params={"interval": 0.1})
sr_pre = nest.Create("spike_recorder")
sr_post = nest.Create("spike_recorder")
nest.Connect(voltmeter, post_pop)
nest.Connect(pre_pop, sr_pre)
nest.Connect(post_pop, sr_post)
nest.Simulate(t_sim)

vm_ev = voltmeter.get("events")
pre_ev = sr_pre.get("events")
post_ev = sr_post.get("events")

fig, axes = plt.subplots(len(pre_currents), 1, figsize=(9, 8), sharex=True)
for ax, gid_pre, gid_post in zip(axes, pre_pop.tolist(), post_pop.tolist()):
    mask = vm_ev["senders"] == gid_post
    pre_spikes = pre_ev["times"][pre_ev["senders"] == gid_pre]
    post_spikes = post_ev["times"][post_ev["senders"] == gid_post]
    rate_pre = len(pre_spikes) / (t_sim / 1000.0)

    ax.plot(vm_ev["times"][mask], vm_ev["V_m"][mask], lw=0.8)
    ax.vlines(pre_spikes, E_L - 4, E_L - 1, color="tab:green", lw=0.8)   # presynaptic spikes below
    ax.vlines(post_spikes, V_th, V_th + 15, color="k", lw=0.8)
    ax.axhline(V_th, ls="--", c="grey", lw=0.8)
    ax.set_ylim(E_L - 5, V_th + 17)
    ax.set_ylabel("V_m (mV)")
    ax.set_title(f"presynaptic rate ≈ {rate_pre:.0f} Hz -> {len(post_spikes)} postsynaptic spikes", fontsize=9)
axes[-1].set_xlabel("time (ms)")
plt.tight_layout()
plt.show()

## Plastic synapses: measuring the STDP window

With a `stdp_synapse`, the weight changes depending on the relative timing $\Delta t = t_{post} - t_{pre}$ of pre- and postsynaptic spikes. To control spike times exactly, we use `parrot_neuron`s, which simply repeat the spikes they receive from a `spike_generator`.

For each $\Delta t$ we build an independent pre/post pair and repeat the pairing 60 times at 1 Hz. The plastic connection enters the postsynaptic parrot on `receptor_type=1`, so that its spikes are *not* repeated (they only drive plasticity). At the end we read the weights with `GetConnections`.

Note: NEST treats the synaptic delay as dendritic, so the post spike must be scheduled at $t_{pre} + d + \Delta t$. A final extra presynaptic spike is added because weight updates are computed when a presynaptic spike arrives.

In [ ]:
nest.ResetKernel()
nest.resolution = 0.1

dt_values = np.arange(-50.0, 51.0, 5.0)   # ms, t_post - t_pre
n_pairings = 60
period = 1000.0                           # ms  (1 Hz pairing)
d_syn = 1.0                               # ms
w0 = 50.0

pre_parrots = nest.Create("parrot_neuron", len(dt_values))
post_parrots = nest.Create("parrot_neuron", len(dt_values))

for i in range(len(dt_values)):
    t_pre = 100.0 + np.arange(n_pairings) * period
    t_post = t_pre + d_syn + dt_values[i]
    t_pre = np.append(t_pre, t_pre[-1] + period)          # flush the last update

    sg_pre = nest.Create("spike_generator", params={"spike_times": t_pre})
    sg_post = nest.Create("spike_generator", params={"spike_times": t_post})
    nest.Connect(sg_pre, pre_parrots[i], syn_spec={"delay": 1.0})
    nest.Connect(sg_post, post_parrots[i], syn_spec={"delay": 1.0})
    nest.Connect(pre_parrots[i], post_parrots[i],
                 syn_spec={"synapse_model": "stdp_synapse", "weight": w0, "delay": d_syn,
                           "Wmax": 100.0, "lambda": 0.01, "alpha": 1.0, "tau_plus": 20.0,
                           "receptor_type": 1})

nest.Simulate(100.0 + (n_pairings + 2) * period)

final_weights = []
for i in range(len(dt_values)):
    c = nest.GetConnections(source=pre_parrots[i], target=post_parrots[i], synapse_model="stdp_synapse")
    final_weights.append(c.weight)
final_weights = np.array(final_weights)

plt.figure(figsize=(6, 4))
plt.plot(dt_values, 100 * (final_weights - w0) / w0, "o-")
plt.axhline(0, c="grey", lw=0.5); plt.axvline(0, c="grey", lw=0.5)
plt.xlabel("Δt = t_post − t_pre (ms)")
plt.ylabel("weight change (%)")
plt.title(f"STDP window after {n_pairings} pairings")
plt.show()